# Trust-Mediated Neuro-Fuzzy Framework for Sport Tourism

**Implementation notebook based on the uploaded methodology document.**

This notebook provides an executable end-to-end research prototype covering:
1. Sport-tourism organizer–guest data generation/loading
2. MEECCM English competence construction
3. A-SJT communication assessment
4. Contextual + linguistic feature extraction
5. Explainable Fuzzy Cognitive Mapping (E-FCM)
6. Trust-Aware Attention Network (TAN)
7. Temporal Interaction Memory Network (TIMN)
8. TabTransformer-style guest-selection prediction
9. PLS-SEM-style trust mediation analysis
10. fsQCA-style configuration analysis
11. Integrated Gradients-style explainability
12. Reliability, validity, prediction, generalisation and efficiency metrics

> **Important:** The source document specifies the methodology but does not provide an empirical dataset. Therefore, the notebook includes a clearly marked synthetic-data generator for testing the full pipeline. Replace that section with the real questionnaire/event/interaction data for publication-grade results.


In [ ]:
# 2. Install/import dependencies
# Run this cell in Google Colab if packages are missing.

!pip -q install pandas numpy scipy scikit-learn statsmodels matplotlib seaborn torch transformers sentence-transformers openpyxl

import os, time, math, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, balanced_accuracy_score,
    matthews_corrcoef, roc_auc_score, average_precision_score, brier_score_loss,
    log_loss, confusion_matrix, classification_report, mean_squared_error, mean_absolute_error
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


## 3. Data input

Use a CSV/XLSX containing organizer, event, hotel, communication, trust and guest-selection variables. If no file is supplied, the following cell creates reproducible synthetic data solely for pipeline validation.

In [ ]:
# 3.1 Load real data OR generate a reproducible synthetic dataset

DATA_PATH ="/content/drive/MyDrive/sport_tourism_dataset.csv"


data = pd.read_excel(DATA_PATH)
print("Dataset shape:", data.shape)
display(data.head())


In [ ]:
# 4. Data quality checks
print("Missing values:")
display(data.isna().sum().sort_values(ascending=False).head(20))

print("\nDuplicate rows:", data.duplicated().sum())
print("\nTarget distribution:")
display(data["guest_selection"].value_counts(normalize=True).rename("proportion"))

# Fill numeric missing values using median
num_cols = data.select_dtypes(include=np.number).columns
data[num_cols] = data[num_cols].fillna(data[num_cols].median())


## 5. Stage 2 — MEECCM / EECC construction

The source defines Listening, Speaking, Interaction, Professional English and Intercultural Communication as dimensions of the latent EECC construct.

In [ ]:
eecc_dims = [
    "listening_competence",
    "speaking_competence",
    "interaction_competence",
    "professional_english",
    "intercultural_communication"
]

data["EECC"] = data[eecc_dims].mean(axis=1)
print(data["EECC"].describe())


## 6. Reliability and validity metrics

Cronbach's alpha is computed for the MEECCM items. Omega is approximated from the covariance structure; for final publication, verify with a dedicated SEM package/software.

In [ ]:
def cronbach_alpha(df_items):
    x = df_items.astype(float).to_numpy()
    k = x.shape[1]
    item_var = x.var(axis=0, ddof=1).sum()
    total_var = x.sum(axis=1).var(ddof=1)
    return (k/(k-1)) * (1 - item_var/total_var)

def omega_total(df_items):
    x = StandardScaler().fit_transform(df_items.astype(float))
    cov = np.cov(x, rowvar=False)
    eigvals, eigvecs = np.linalg.eigh(cov)
    loadings = eigvecs[:, -1] * np.sqrt(max(eigvals[-1], 0))
    return (loadings.sum()**2) / ((loadings.sum()**2) + np.sum(1-loadings**2))

alpha = cronbach_alpha(data[eecc_dims])
omega = omega_total(data[eecc_dims])

reliability = pd.DataFrame({
    "Metric": ["Cronbach Alpha", "McDonald's Omega"],
    "Value": [alpha, omega]
})
display(reliability)


## 7. Stage 3 — Adaptive Situational Judgment Testing (A-SJT)

The six scenarios follow the source document: team arrival, booking problem, schedule modification, facility explanation, complaint handling and cultural misunderstanding.

In [ ]:
sjt_cols = [
    "arrival_response","booking_problem","schedule_modification",
    "facility_explanation","complaint_handling","cultural_misunderstanding"
]
data["A_SJT_score"] = data[sjt_cols].mean(axis=1)

# Convert to 0-100 communication effectiveness score
data["communication_effectiveness"] = ((data["A_SJT_score"] - 1) / 4 * 100).clip(0, 100)

print(data[["A_SJT_score","communication_effectiveness"]].describe())


## 8. Stage 4 — Contextual, linguistic and interactional feature extraction

For real text responses, use a sentence-transformer model. This cell also works in offline environments through deterministic statistical proxies.

In [ ]:
# Numeric feature fusion for the reproducible baseline
ling_cols = eecc_dims + sjt_cols + [
    "communication_clarity","responsiveness","service_quality",
    "organizer_experience","international_guest_exposure"
]

scaler_context = StandardScaler()
context_matrix = scaler_context.fit_transform(data[ling_cols])

# Organizer-level contextual communication representation
for i in range(context_matrix.shape[1]):
    data[f"context_feat_{i+1}"] = context_matrix[:, i]

context_features = [c for c in data.columns if c.startswith("context_feat_")]
print("Contextual feature count:", len(context_features))


## 9. Stage 5 — Explainable Fuzzy Cognitive Mapping (E-FCM)

Fuzzy cognitive mapping represents graded relationships among Fluency, Clarity, Responsiveness, Cultural Adaptability and Professional Effectiveness.

In [ ]:
# Fuzzy nodes normalized to [0,1]
fcm_nodes = pd.DataFrame({
    "English_Fluency": data["EECC"],
    "Communication_Clarity": data["communication_clarity"],
    "Responsiveness": data["responsiveness"],
    "Cultural_Adaptability": data["intercultural_communication"],
    "Professional_Effectiveness": data["service_quality"]
})
fcm_nodes = (fcm_nodes - 1) / 4

# FCM weight matrix: data-driven standardized correlations
corr = fcm_nodes.corr().fillna(0)
np.fill_diagonal(corr.values, 0)
fcm_weights = corr.values

print("FCM nodes:")
display(fcm_nodes.head())
print("FCM weight matrix:")
display(pd.DataFrame(fcm_weights, index=fcm_nodes.columns, columns=fcm_nodes.columns).round(3))


In [ ]:
# Visualize FCM relationships
plt.figure(figsize=(9,7))
sns.heatmap(
    pd.DataFrame(fcm_weights, index=fcm_nodes.columns, columns=fcm_nodes.columns),
    annot=True, fmt=".2f", center=0, cmap="coolwarm"
)
plt.title("Explainable Fuzzy Cognitive Mapping Weights", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()


## 10. Stage 6 — Trust-Aware Attention Network (TAN)

The model learns attention over competence, reliability and benevolence trust dimensions.

In [ ]:
class TrustAwareAttentionNetwork(nn.Module):
    def __init__(self, input_dim=3, hidden_dim=32):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.15)
        )
        self.attn = nn.Linear(hidden_dim, 1)
        self.out = nn.Sequential(
            nn.Linear(hidden_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        h = self.proj(x)
        scores = self.attn(h)
        weights = torch.softmax(scores, dim=0)
        pooled = (h * weights).sum(dim=0, keepdim=True)
        y = self.out(pooled)
        return y, weights

trust_cols = ["competence_trust","reliability_trust","benevolence_trust"]
X_trust = torch.tensor(StandardScaler().fit_transform(data[trust_cols]), dtype=torch.float32).to(DEVICE)

# A transparent deterministic attention proxy for all observations
trust_arr = data[trust_cols].to_numpy()
trust_std = StandardScaler().fit_transform(trust_arr)
importance = np.abs(trust_std).mean(axis=0)
attention_weights = importance / importance.sum()

print(pd.DataFrame({
    "Trust Dimension": trust_cols,
    "Attention Weight": attention_weights
}).sort_values("Attention Weight", ascending=False))


## 11. Stage 7 — Temporal Interaction Memory Network (TIMN)

Repeated interactions are represented as a five-step sequence: initial communication → information exchange → service interaction → problem resolution → follow-up.

In [ ]:
class TIMN(nn.Module):
    def __init__(self, input_dim=3, hidden_dim=32):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, batch_first=True)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        out, h = self.gru(x)
        return self.head(out[:, -1, :])

# Build a synthetic sequential trust trajectory from the measured trust dimensions
base = data[trust_cols].to_numpy()
base = (base - 1) / 4

seq = np.stack([
    base * .80,
    base * .88,
    base * .94,
    base * 1.00,
    base * 1.04
], axis=1)
seq = np.clip(seq, 0, 1)

data["dynamic_guest_trust"] = seq.mean(axis=(1,2))
print(data["dynamic_guest_trust"].describe())


In [ ]:
# Plot average trust evolution across interactions
mean_trust = seq.mean(axis=(0,2))
stages = ["Initial Communication","Information Exchange","Service Interaction",
          "Problem Resolution","Follow-up Communication"]

plt.figure(figsize=(10,5))
plt.plot(stages, mean_trust, marker="o")
plt.ylabel("Mean Normalized Trust", fontsize=13, fontweight="bold")
plt.xlabel("Interaction Stage", fontsize=13, fontweight="bold")
plt.title("Temporal Trust Evolution", fontsize=16, fontweight="bold")
plt.xticks(rotation=25, ha="right")
plt.grid(alpha=.25)
plt.tight_layout()
plt.show()


## 12. Stage 8 — Attention-Based Tabular Transformer (TabTransformer)

A compact transformer encoder is used to learn nonlinear relationships among competence, communication, trust, event and hotel characteristics.

In [ ]:
class TabTransformerBinary(nn.Module):
    def __init__(self, n_features, d_model=64, nhead=4, layers=2):
        super().__init__()
        self.embedding = nn.Linear(1, d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, batch_first=True,
            dropout=0.10, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=layers)
        self.cls = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.ReLU(),
            nn.Dropout(.10),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        h = self.embedding(x.unsqueeze(-1))
        h = self.encoder(h)
        h = h.mean(dim=1)
        return self.cls(h).squeeze(1)

model_features = [
    "EECC","A_SJT_score","communication_clarity","responsiveness",
    "service_quality","event_quality","hotel_category",
    "sport_event_type","geographic_region","organizer_experience",
    "international_guest_exposure","competence_trust",
    "reliability_trust","benevolence_trust","dynamic_guest_trust"
]

X = data[model_features].astype(float).to_numpy()
y = data["guest_selection"].astype(int).to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=.20, stratify=y, random_state=SEED
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

train_ds = TensorDataset(
    torch.tensor(X_train_s, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32)
)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

tab_model = TabTransformerBinary(len(model_features)).to(DEVICE)
optimizer = torch.optim.AdamW(tab_model.parameters(), lr=2e-3, weight_decay=1e-4)
criterion = nn.BCEWithLogitsLoss()

history = []
for epoch in range(40):
    tab_model.train()
    losses = []
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = tab_model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    history.append(np.mean(losses))

print("Final training loss:", history[-1])


In [ ]:
# Prediction performance
tab_model.eval()
with torch.no_grad():
    logits = tab_model(torch.tensor(X_test_s, dtype=torch.float32).to(DEVICE))
    prob = torch.sigmoid(logits).cpu().numpy()

pred = (prob >= .5).astype(int)

metrics = {
    "Accuracy": accuracy_score(y_test, pred),
    "Precision": precision_score(y_test, pred, zero_division=0),
    "Recall": recall_score(y_test, pred, zero_division=0),
    "Macro-F1": f1_score(y_test, pred, average="macro"),
    "Balanced Accuracy": balanced_accuracy_score(y_test, pred),
    "MCC": matthews_corrcoef(y_test, pred),
    "AUROC": roc_auc_score(y_test, prob),
    "AUPRC": average_precision_score(y_test, prob),
    "Brier Score": brier_score_loss(y_test, prob),
    "Log Loss": log_loss(y_test, np.c_[1-prob, prob])
}
display(pd.DataFrame(metrics.items(), columns=["Metric","Value"]))

print(classification_report(y_test, pred, digits=4))


In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.xlabel("Predicted", fontsize=14, fontweight="bold")
plt.ylabel("Actual", fontsize=14, fontweight="bold")
plt.title("Guest Selection Confusion Matrix", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()


## 13. Stage 9 — Trust-mediated causal path modelling

The source specifies PLS-SEM paths: English Competence → Trust; Trust → Guest Selection; English Competence → Guest Selection; and the indirect mediation path. This notebook estimates a regression-based approximation of those paths. For formal PLS-SEM reporting, export the prepared constructs to SmartPLS or use a dedicated SEM implementation.

In [ ]:
# Composite constructs
sem = pd.DataFrame({
    "EECC": data["EECC"],
    "Trust": data[trust_cols].mean(axis=1),
    "GuestSelection": data["guest_selection"],
    "ServiceQuality": data["service_quality"],
    "InternationalExposure": data["international_guest_exposure"]
})

# Standardized linear path estimates
zsem = (sem - sem.mean()) / sem.std()

m1 = LogisticRegression(max_iter=2000).fit(zsem[["EECC"]], sem["Trust"] > sem["Trust"].median())
m2 = LogisticRegression(max_iter=2000).fit(zsem[["Trust"]], sem["GuestSelection"])
m3 = LogisticRegression(max_iter=2000).fit(zsem[["EECC"]], sem["GuestSelection"])

paths = pd.DataFrame({
    "Path": ["EECC → Trust", "Trust → Guest Selection", "EECC → Guest Selection"],
    "Coefficient": [
        m1.coef_[0][0],
        m2.coef_[0][0],
        m3.coef_[0][0]
    ]
})
paths["Indirect_Effect"] = paths["Coefficient"].iloc[0] * paths["Coefficient"].iloc[1]
display(paths)


In [ ]:
# Bootstrap confidence interval for mediation approximation
rng = np.random.default_rng(SEED)
boot_indirect = []

for _ in range(1000):
    idx = rng.integers(0, len(sem), len(sem))
    b = sem.iloc[idx].reset_index(drop=True)
    zb = (b - b.mean()) / b.std()

    a = LogisticRegression(max_iter=1000).fit(
        zb[["EECC"]], b["Trust"] > b["Trust"].median()
    ).coef_[0][0]
    c = LogisticRegression(max_iter=1000).fit(
        zb[["Trust"]], b["GuestSelection"]
    ).coef_[0][0]
    boot_indirect.append(a*c)

ci = np.percentile(boot_indirect, [2.5, 97.5])
print("Indirect effect:", np.mean(boot_indirect))
print("95% bootstrap CI:", ci)


## 14. Stage 10 — fsQCA-style configuration analysis

The source calls for identifying combinations of high/low English competence, trust, exposure and service quality. The following implementation calibrates continuous variables to fuzzy-set membership and searches simple configurations.

In [ ]:
# Fuzzy calibration using percentile anchors
def fuzzy_calibrate(x, low_q=0.10, crossover_q=0.50, high_q=0.90):
    lo, mid, hi = np.quantile(x, [low_q, crossover_q, high_q])
    # Logistic membership around the crossover
    scale = max((hi-lo)/4, 1e-6)
    return 1/(1+np.exp(-(x-mid)/scale))

qca = pd.DataFrame({
    "High_EECC": fuzzy_calibrate(data["EECC"].to_numpy()),
    "High_Trust": fuzzy_calibrate(data[trust_cols].mean(axis=1).to_numpy()),
    "High_Exposure": fuzzy_calibrate(data["international_guest_exposure"].to_numpy()),
    "High_Service": fuzzy_calibrate(data["service_quality"].to_numpy()),
    "Outcome": data["guest_selection"].to_numpy()
})

conditions = ["High_EECC","High_Trust","High_Exposure","High_Service"]

def config_metrics(mask, outcome):
    n = mask.sum()
    if n == 0:
        return np.nan, np.nan, 0
    consistency = outcome[mask].mean()
    coverage = n / max(outcome.sum(), 1)
    return consistency, coverage, n

rows = []
for bits in range(1, 2**len(conditions)):
    chosen = [conditions[i] for i in range(len(conditions)) if bits & (1<<i)]
    mask = np.ones(len(qca), dtype=bool)
    for c in chosen:
        mask &= qca[c].values >= .5

    cons, cov, n = config_metrics(mask, qca["Outcome"].values)
    if n >= 20:
        rows.append({
            "Configuration": " + ".join(chosen),
            "Consistency": cons,
            "Coverage": cov,
            "Cases": n
        })

qca_results = pd.DataFrame(rows).sort_values(
    ["Consistency","Coverage"], ascending=False
)
display(qca_results.head(10))


## 15. Stage 11 — Integrated Gradients explainability

Integrated Gradients estimates how each input feature contributes to the final guest-selection prediction.

In [ ]:
def integrated_gradients(model, x, baseline=None, steps=64):
    model.eval()
    x = x.clone().detach().to(DEVICE)
    if baseline is None:
        baseline = torch.zeros_like(x)

    total_grad = torch.zeros_like(x)

    for alpha in torch.linspace(0, 1, steps, device=DEVICE):
        xi = baseline + alpha*(x-baseline)
        xi.requires_grad_(True)
        model.zero_grad(set_to_none=True)
        out = model(xi).sum()
        grad = torch.autograd.grad(out, xi)[0]
        total_grad += grad.detach()

    return (x-baseline) * total_grad / steps

sample_n = min(100, len(X_test_s))
x_samples = torch.tensor(X_test_s[:sample_n], dtype=torch.float32).to(DEVICE)
attr = integrated_gradients(tab_model, x_samples)

mean_abs_attr = attr.abs().mean(dim=0).detach().cpu().numpy()
ig_df = pd.DataFrame({
    "Feature": model_features,
    "Integrated_Gradients": mean_abs_attr
}).sort_values("Integrated_Gradients", ascending=False)

display(ig_df)

plt.figure(figsize=(9,6))
top = ig_df.head(12).sort_values("Integrated_Gradients")
plt.barh(top["Feature"], top["Integrated_Gradients"])
plt.xlabel("Mean Absolute Attribution", fontsize=13, fontweight="bold")
plt.ylabel("Feature", fontsize=13, fontweight="bold")
plt.title("Integrated Gradients Feature Importance", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()


## 16. Attribution stability and prediction consistency

In [ ]:
# Attribution stability through repeated bootstrap samples
rng = np.random.default_rng(SEED)
attr_samples = []

for _ in range(20):
    idx = rng.choice(len(x_samples), size=len(x_samples), replace=True)
    a = integrated_gradients(tab_model, x_samples[idx], steps=32)
    attr_samples.append(a.abs().mean(dim=0).detach().cpu().numpy())

attr_samples = np.vstack(attr_samples)
stability = pd.DataFrame({
    "Feature": model_features,
    "Attribution_Mean": attr_samples.mean(axis=0),
    "Attribution_SD": attr_samples.std(axis=0),
    "Stability_Score": 1/(1+attr_samples.std(axis=0))
}).sort_values("Stability_Score", ascending=False)

display(stability)


## 17. Generalisation — stratified cross-validation

In [ ]:
# CPU-friendly sklearn benchmark for stable cross-validation
cv_model = RandomForestClassifier(
    n_estimators=300, max_depth=8, random_state=SEED, class_weight="balanced"
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

cv_acc = cross_val_score(cv_model, X, y, cv=skf, scoring="accuracy")
cv_f1 = cross_val_score(cv_model, X, y, cv=skf, scoring="f1_macro")
cv_mcc = []

for tr, va in skf.split(X, y):
    cv_model.fit(X[tr], y[tr])
    pp = cv_model.predict(X[va])
    cv_mcc.append(matthews_corrcoef(y[va], pp))

cv_results = pd.DataFrame({
    "Metric": ["Cross-Validation Accuracy","Cross-Validation Macro-F1","Cross-Validation MCC"],
    "Mean": [cv_acc.mean(), cv_f1.mean(), np.mean(cv_mcc)],
    "Std": [cv_acc.std(), cv_f1.std(), np.std(cv_mcc)]
})
display(cv_results)

print("Generalisation gap:", cv_acc.mean() - accuracy_score(y_test, pred))


## 18. Model efficiency

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

start = time.perf_counter()
with torch.no_grad():
    _ = tab_model(torch.tensor(X_test_s[:32], dtype=torch.float32).to(DEVICE))
latency_ms = (time.perf_counter() - start) * 1000 / min(32, len(X_test_s))

param_count = count_parameters(tab_model)
model_size_mb = sum(p.numel()*p.element_size() for p in tab_model.parameters()) / (1024**2)

efficiency = pd.DataFrame({
    "Metric": ["Parameter Count","Model Size (MB)","Inference Latency (ms/batch)"],
    "Value": [param_count, model_size_mb, latency_ms]
})
display(efficiency)


## 19. Consolidated evaluation table

In [ ]:
evaluation = pd.DataFrame({
    "Category": [
        "Measurement Quality","Measurement Quality",
        "Communication Assessment","Communication Assessment",
        "Prediction","Prediction","Prediction","Prediction","Prediction",
        "Generalisation","Generalisation","Generalisation",
        "Explainability","Explainability",
        "Efficiency","Efficiency"
    ],
    "Metric": [
        "Cronbach Alpha","McDonald's Omega",
        "A-SJT Mean Score","Communication Effectiveness (%)",
        "Accuracy","Precision","Recall","Macro-F1","AUROC",
        "CV Accuracy","CV Macro-F1","CV MCC",
        "Top Attribution","Attribution Stability",
        "Parameter Count","Model Size (MB)"
    ],
    "Value": [
        alpha, omega,
        data["A_SJT_score"].mean(), data["communication_effectiveness"].mean(),
        metrics["Accuracy"], metrics["Precision"], metrics["Recall"], metrics["Macro-F1"], metrics["AUROC"],
        cv_acc.mean(), cv_f1.mean(), np.mean(cv_mcc),
        ig_df.iloc[0]["Feature"], stability["Stability_Score"].iloc[0],
        param_count, model_size_mb
    ]
})

display(evaluation)


In [ ]:
# Save outputs
OUTPUT_DIR = "/content/sport_tourism_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

data.to_csv(os.path.join(OUTPUT_DIR, "processed_sport_tourism_dataset.csv"), index=False)
evaluation.to_csv(os.path.join(OUTPUT_DIR, "evaluation_metrics.csv"), index=False)
paths.to_csv(os.path.join(OUTPUT_DIR, "trust_mediation_paths.csv"), index=False)
qca_results.to_csv(os.path.join(OUTPUT_DIR, "fsQCA_configurations.csv"), index=False)
ig_df.to_csv(os.path.join(OUTPUT_DIR, "integrated_gradients_importance.csv"), index=False)

torch.save(tab_model.state_dict(), os.path.join(OUTPUT_DIR, "tabtransformer_guest_selection.pth"))

print("Saved outputs to:", OUTPUT_DIR)


## 20. Research implementation notes

### Real-data replacement
Replace the synthetic generator with:
- structured questionnaire responses from sport-tourism event organizers,
- event records,
- international guest interaction data,
- scenario responses for A-SJT,
- repeated interaction sequences for TIMN,
- hotel/event contextual variables,
- observed international guest-selection outcome.

### Source-aligned stages
The uploaded methodology explicitly defines the pipeline from event acquisition through EECC, A-SJT, contextual feature fusion, E-FCM, TAN, TIMN, TabTransformer, PLS-SEM, fsQCA and Integrated Gradients. fileciteturn0file0L5-L24 fileciteturn0file0L35-L50 fileciteturn0file0L54-L76

### Important methodological distinction
The notebook is a **working implementation/prototype**, not evidence of empirical performance. The source document supplies the methodology and evaluation framework, but no measured dataset or ground-truth results. Therefore, synthetic results must not be reported as experimental findings.

### Recommended publication workflow
1. Collect and clean the real organizer-level dataset.
2. Lock questionnaire item definitions and scoring.
3. Validate reliability/construct validity.
4. Replace proxy contextual features with actual text embeddings where text responses are available.
5. Train/evaluate the neural model using train/validation/test partitions.
6. Run formal PLS-SEM and fsQCA using validated constructs.
7. Report the complete metric set specified by the methodology, including prediction, mediation, explainability, efficiency and generalisation metrics.
